# Tarea · Diagnóstico de cáncer de mama con cinco clasificadores
### Regresión logística · Naïve Bayes · SVM · Árbol de decisión · Random Forest

**FP13 · Inteligencia Artificial** — Universidad Autónoma de Guadalajara

**Unidad 5:** 5.1 Clasificación · 5.2 Aprendizaje supervisado · 5.3 Preprocesamiento · 5.4 Codificación de etiquetas · 5.5 Regresión logística · 5.6 Naïve Bayes · 5.7 Matrices de confusión · 5.8 SVM
**Unidad 6:** 6.1 Árboles de decisión · 6.3 Bosques aleatorios · 6.5 Búsqueda de rejilla

---

### 🏥 El problema clínico
Cuando se detecta una masa en la mama, una forma rápida y poco invasiva de estudiarla es la **biopsia por aspiración con aguja fina (BAAF)**: con una aguja delgada se extraen células, se tiñen y se observan al microscopio. En la Universidad de Wisconsin se digitalizaron esas imágenes y se midieron los **núcleos de las células** (tamaño, forma, textura…).

Un servicio de patología quiere una herramienta de **apoyo** que, con esas 30 mediciones, indique si el tumor **probablemente es maligno** para priorizar el caso.

> ⚠️ Un **falso negativo** (decir *benigno* a un tumor *maligno*) significa que una paciente con cáncer se va a casa sin tratamiento. Por eso, en esta tarea **la exactitud no basta**: buscaremos modelos que **casi no dejen escapar tumores malignos**.

### 📋 Instrucciones
1. Trabaja en tu entorno de github (VS Code) y ejecuta las celdas **en orden**, de arriba hacia abajo.
2. Cada `____` es un hueco que debes completar. Las líneas con huecos están marcadas con ✏️. **No modifiques el resto del código.**
3. Si una celda marca error, revisa los huecos de esa celda y de las anteriores.
4. Al final responde la **reflexión** en las celdas marcadas con ✍️.

### 🧮 Puntuación
| Sección | Puntos |
|---|:---:|
| 1–3 · Carga, exploración y preprocesamiento | 15 |
| 4–8 · Cinco modelos con búsqueda de rejilla (9 c/u) | 45 |
| 9–10 · Comparación final y ajuste del umbral | 10 |
| 11 · Reflexión: ¿cuál es el mejor modelo? | 30 |
| **Total** | **100** |

### ¿Qué cambia respecto a la práctica de clase?
- La etiqueta viene como **texto** (`M` / `B`) → usaremos **`LabelEncoder`** (tema 5.4).
- Hiperparámetros nuevos: kernel `"poly"` con `degree` (SVM), `min_samples_split` (árbol), `max_features="log2"` y `min_samples_leaf` (bosque), y `np.logspace` para generar los valores de `var_smoothing` (Naïve Bayes).

---
## 0 · Librerías

In [ ]:
# Importando librerías
import warnings
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la pantalla quede limpia

import numpy as np                     # cálculo numérico
import pandas as pd                    # tablas de datos
import matplotlib.pyplot as plt        # gráficas

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

---
## 1 · Los datos  *(5 puntos)*

**Breast Cancer Wisconsin (Diagnostic)** — UCI Machine Learning, publicado en Kaggle como `uciml/breast-cancer-wisconsin-data`. El archivo dentro del dataset se llama **`data.csv`**.

569 pacientes. A cada núcleo celular se le midieron **10 características**:

| Grupo | Variables | ¿Qué describe? |
|---|---|---|
| Tamaño | `radius`, `perimeter`, `area` | Qué tan grande es el núcleo |
| Textura | `texture`, `smoothness` | Variación de tonos de gris · qué tan lisa es la orilla |
| Forma | `compactness`, `concavity`, `concave points`, `symmetry`, `fractal_dimension` | Qué tan irregular o "hundido" es el contorno |

Cada característica aparece **3 veces**: `_mean` (promedio de los núcleos), `_se` (error estándar) y `_worst` (promedio de los 3 núcleos más grandes/anormales) → **30 variables**.

La columna **`diagnosis`** es lo que queremos predecir: **`M` = maligno**, **`B` = benigno**.

In [ ]:
# Importando datos desde Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                            "uciml/breast-cancer-wisconsin-data",
                            "data.csv")

In [ ]:
# Explorando las primeras filas
df.head()    # ✏️

In [ ]:
# Imprimiendo renglones y columnas del dataframe
df.shape

---
## 2 · Exploración  *(4 puntos)*

In [ ]:
# Comprobando si existen datos nulos
df.____().sum()    # ✏️

Observa la última columna, **`Unnamed: 32`**: ¡está vacía en los 569 renglones! Aparece porque el archivo CSV trae una coma de más al final de cada línea. No aporta nada, así que la eliminaremos junto con **`id`** (un número de expediente no ayuda a diagnosticar).

In [ ]:
# Proporción de cada diagnóstico
df['diagnosis'].____(normalize=True)    # ✏️

In [ ]:
# Comparando la escala de tres variables
df[['area_mean', 'radius_mean', 'smoothness_mean']].describe().T

`area_mean` llega a **~2,500**, mientras que `smoothness_mean` no pasa de **0.17**. La regresión logística y la SVM comparan variables con coeficientes y distancias, así que necesitan que todas estén en la **misma escala** → `StandardScaler` (tema 5.3). Los árboles no: solo preguntan *"¿es mayor que…?"*.

---
## 3 · Preprocesamiento y codificación de etiquetas  *(6 puntos)*

1. **Eliminar** las columnas que no sirven para predecir (`id`, `Unnamed: 32`).
2. **Codificar la etiqueta** con `LabelEncoder`: los modelos solo entienden números.
3. **Separar** `X` (las 30 mediciones) de `y` (el diagnóstico).
4. **Dividir** en entrenamiento y prueba conservando la proporción de malignos (`stratify`).

In [ ]:
# Eliminando las columnas que no sirven para predecir
df = df.drop(columns=[____, ____])    # ✏️
df.shape

In [ ]:
# Codificando la etiqueta: texto → número
codificador = LabelEncoder()
df['diagnosis'] = codificador.____(df['diagnosis'])    # ✏️

# ¿Qué número recibió cada letra?
print(codificador.classes_)          # posición 0 → primera letra, posición 1 → segunda letra
df['diagnosis'].value_counts()

`LabelEncoder` asigna los números en **orden alfabético**: **B → 0** y **M → 1**. ¡Justo lo que queremos! La clase **positiva (1)** es la que buscamos detectar: **maligno**. Así, `recall` medirá exactamente *"de cada 100 tumores malignos, ¿cuántos detecta el modelo?"*.

> ¿Por qué `LabelEncoder` y no one-hot? Porque es la **variable objetivo**: una sola columna con dos categorías. El one-hot se usa para variables de **entrada** con texto.

In [ ]:
# Separando X (mediciones) de y (diagnóstico)
y = df[____]    # ✏️
X = df.drop(columns=['diagnosis'])

print(X.shape, y.shape)

In [ ]:
# Dividiendo en entrenamiento (80 %) y prueba (20 %), conservando la proporción B/M
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,
                                                    stratify=____, random_state=42)    # ✏️

print(f'Dimensiones de entrenamiento: {X_train.shape}')
print(f'Dimensiones de prueba:        {X_test.shape}')
print(f'Tumores malignos en prueba:   {y_test.sum()}')

---
## 4 · ¿Cómo vamos a evaluar?

Las filas son la **realidad** y las columnas lo que **predice** el modelo:

| | Predice **Benigno** | Predice **Maligno** |
|---|:---:|:---:|
| **Realmente benigno** | ✅ VN · benigno bien descartado | ⚠️ FP · falsa alarma (estudios de más) |
| **Realmente maligno** | ❌ **FN · cáncer que se va a casa** | ✅ VP · cáncer detectado |

<br>

| Métrica | Pregunta clínica que responde |
|---|---|
| **Exactitud** (`accuracy`) | De cada 100 pacientes, ¿cuántas clasifica bien? |
| **Precisión** (`precision`) | Cuando dice "maligno", ¿cuántas veces acierta? |
| **Sensibilidad** (`recall`) | De cada 100 tumores malignos, ¿cuántos detecta? |
| **Especificidad** | De cada 100 tumores benignos, ¿cuántos descarta bien? |

**Regla de esta tarea:** cada modelo se optimiza con 3 métricas distintas (`accuracy`, `precision` y `recall`). De esas 3 versiones, guardaremos como **finalista** la optimizada con **`recall`** (sensibilidad), porque es la que busca dejar escapar el menor número de tumores malignos (FN).

In [ ]:
# Nombres de las clases (0 y 1 respectivamente)
clases = ['Benigno', 'Maligno']

# Métricas con las que vamos a optimizar cada modelo
metricas = ['accuracy', 'precision', 'recall']

# Aquí guardaremos el modelo finalista de cada algoritmo
finalistas = {}

---
## 5 · Regresión logística  *(9 puntos)*

| Hiperparámetro | Qué controla | Valores que probaremos |
|---|---|---|
| **`C`** | Inverso de la regularización: `C` pequeño = modelo más simple y "prudente" | 0.01 · 0.1 · 1 · 10 · 100 |
| **`l1_ratio`** | Tipo de regularización: 0 = L2 · 1 = L1 (puede apagar variables) · 0.5 = mezcla | 0 · 0.5 · 1 |
| **`class_weight`** | `"balanced"` le da más peso a la clase con menos pacientes (malignos) | `None` · `"balanced"` |

In [ ]:
# Importando el modelo
from sklearn.linear_model import LogisticRegression

# Pipeline: escalar → modelo
reglog_base = Pipeline([
    ("escalar", ____),    # ✏️
    ("modelo",  ____(penalty="elasticnet", solver="saga", l1_ratio=0,    # ✏️
                                   max_iter=5000, random_state=42))
])

# Ajustando el modelo con valores por defecto
reglog_base.____(X_train, y_train)    # ✏️

In [ ]:
# Calculando las predicciones sobre el conjunto de prueba
y_pred = reglog_base.____(X_test)    # ✏️

# Imprimiendo el reporte de clasificación
print(classification_report(y_test, y_pred, target_names=clases))

# Construyendo la matriz de confusión
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases, cmap='Blues')
plt.title('Regresión logística base')
plt.show()

In [ ]:
# Hiperparámetros a optimizar
rejilla_reglog = {
    "modelo__C":            [0.01, 0.1, 1, 10, 100],
    "modelo__l1_ratio":     [0, 0.5, ____],             # 0 = L2 · 0.5 = mezcla · 1 = L1  ✏️
    "modelo__class_weight": [None, ____],    # ✏️
}
# 5 × 3 × 2 = 30 combinaciones

In [ ]:
mejores_modelos = {}

for metrica in metricas:
    busqueda = GridSearchCV(____, ____, scoring=metrica, cv=5, n_jobs=-1)    # ✏️
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.____    # ✏️
    print(f'{metrica} >> {busqueda.best_score_:.3f}  {busqueda.best_params_}')

In [ ]:
# Mostrando las matrices de confusión de las 3 versiones optimizadas
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))      # 1 fila con 3 recuadros

for eje, metrica in zip(ejes, metricas):
    y_pred = mejores_modelos[metrica].predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases,
                                            cmap='Blues', colorbar=False, ax=eje)
    eje.set_title(f'Objetivo: {metrica}')

plt.suptitle('Regresión logística')
plt.tight_layout()
plt.show()

In [ ]:
# Guardando como finalista la versión optimizada con recall
finalistas['Regresión logística'] = mejores_modelos[____]    # ✏️

---
## 6 · Naïve Bayes  *(9 puntos)*

| Hiperparámetro | Qué controla | Valores que probaremos |
|---|---|---|
| **`var_smoothing`** | Suma una pequeña cantidad a las varianzas para que ninguna sea casi cero. Valores grandes = modelo más "suavizado" | 10 valores entre 10⁻⁹ y 1, generados con `np.logspace(-9, 0, 10)` 🆕 |
| **`priors`** | Probabilidad inicial que el modelo asume para cada clase, en el orden `[benigno, maligno]`. `None` = la proporción de los datos | `None` · `[0.5, 0.5]` · `[0.3, 0.7]` |

<br>

> `np.logspace(-9, 0, 10)` genera 10 números espaciados en potencias de 10: 10⁻⁹, 10⁻⁸, …, 10⁰ = 1. Es muy útil cuando no sabemos ni el orden de magnitud del valor correcto.

In [ ]:
# Importando el modelo
from sklearn.naive_bayes import GaussianNB

# Pipeline: escalar → modelo
nb_base = Pipeline([
    ("escalar", StandardScaler()),
    ("modelo",  ____)    # ✏️
])

# Ajustando el modelo con valores por defecto
nb_base.fit(X_train, y_train)

In [ ]:
# Calculando las predicciones sobre el conjunto de prueba
y_pred = nb_base.predict(X_test)

# Imprimiendo el reporte de clasificación
print(classification_report(____, ____, target_names=clases))    # ✏️

# Construyendo la matriz de confusión
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases, cmap='Blues')
plt.title('Naïve Bayes base')
plt.show()

In [ ]:
# Hiperparámetros a optimizar
rejilla_nb = {
    "modelo__var_smoothing": np.____(-9, 0, 10),               # 10 valores: 1e-9, 1e-8, …, 1  ✏️
    "modelo__priors":        [None, [0.5, 0.5], ____],        # [benigno, maligno]  ✏️
}
# 10 × 3 = 30 combinaciones

In [ ]:
mejores_modelos = {}

for metrica in metricas:
    busqueda = GridSearchCV(____, ____, scoring=____, cv=5, n_jobs=-1)    # ✏️
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.best_estimator_
    print(f'{metrica} >> {busqueda.best_score_:.3f}  {busqueda.best_params_}')

In [ ]:
# Mostrando las matrices de confusión de las 3 versiones optimizadas
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))

for eje, metrica in zip(ejes, metricas):
    y_pred = mejores_modelos[metrica].predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases,
                                            cmap='Blues', colorbar=False, ax=eje)
    eje.set_title(f'Objetivo: {metrica}')

plt.suptitle('Naïve Bayes')
plt.tight_layout()
plt.show()

In [ ]:
# Guardando como finalista la versión optimizada con recall
finalistas['Naïve Bayes'] = mejores_modelos['recall']

---
## 7 · Máquinas de vectores de soporte (SVM)  *(9 puntos)*

| Hiperparámetro | Qué controla | Valores que probaremos |
|---|---|---|
| **`kernel`** | La forma de la frontera: `"linear"` = recta · `"rbf"` = curvas suaves · `"poly"` 🆕 = curva polinomial | los tres |
| **`C`** | Qué tanto castiga los errores en entrenamiento: `C` grande = frontera más ajustada (riesgo de sobreajuste) | 0.01 · 0.1 · 1 · 10 · 100 |
| **`gamma`** *(solo rbf)* | Qué tan "local" es la influencia de cada paciente | `"scale"` · 0.01 · 0.1 |
| **`degree`** 🆕 *(solo poly)* | Grado del polinomio: 2 = parábola, 3 = cúbica | 2 · 3 |
| **`class_weight`** | `"balanced"` da más peso a los malignos | `None` · `"balanced"` |

Como cada kernel usa hiperparámetros distintos, la rejilla es una **lista de diccionarios** (uno por kernel).

In [ ]:
# Importando el modelo
from sklearn.svm import SVC

# Pipeline: escalar → modelo
svm_base = Pipeline([
    ("escalar", StandardScaler()),
    ("modelo",  ____())    # ✏️
])

# Ajustando el modelo con valores por defecto
svm_base.fit(X_train, y_train)

In [ ]:
# Calculando las predicciones sobre el conjunto de prueba
y_pred = svm_base.____(X_test)    # ✏️

# Imprimiendo el reporte de clasificación
print(classification_report(y_test, y_pred, target_names=clases))

# Construyendo la matriz de confusión
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases, cmap='Blues')
plt.title('SVM base')
plt.show()

In [ ]:
# Hiperparámetros a optimizar
valores_C = [0.01, 0.1, 1, 10, 100]

rejilla_svm = [
    {"modelo__kernel":       ["linear"],
     "modelo__C":            valores_C,
     "modelo__class_weight": [None, "balanced"]},

    {"modelo__kernel":       [____],    # ✏️
     "modelo__C":            valores_C,
     "modelo__gamma":        ["scale", 0.01, 0.1],
     "modelo__class_weight": [None, "balanced"]},

    {"modelo__kernel":       ["poly"],                    # NUEVO: kernel polinomial
     "modelo__C":            valores_C,
     ____:     [2, 3],                      # grado del polinomio  ✏️
     "modelo__class_weight": [None, "balanced"]},
]
# 10 + 30 + 20 = 60 combinaciones

In [ ]:
mejores_modelos = {}

for metrica in metricas:
    busqueda = GridSearchCV(____, ____, scoring=metrica, cv=5, n_jobs=-1)    # ✏️
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.best_estimator_
    print(f'{metrica} >> {busqueda.best_score_:.3f}  {busqueda.best_params_}')

In [ ]:
# Mostrando las matrices de confusión de las 3 versiones optimizadas
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))

for eje, metrica in zip(ejes, metricas):
    y_pred = mejores_modelos[metrica].predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases,
                                            cmap='Blues', colorbar=False, ax=eje)
    eje.set_title(f'Objetivo: {metrica}')

plt.suptitle('SVM')
plt.tight_layout()
plt.show()

In [ ]:
# Guardando como finalista la versión optimizada con recall
finalistas['SVM'] = mejores_modelos['recall']

---
## 8 · Árbol de decisión y Random Forest  *(9 + 9 puntos)*

Los árboles **no necesitan escalar**: solo hacen preguntas del tipo *"¿`perimeter_worst` ≤ 112.8?"*, y la respuesta es la misma en cualquier escala. Por eso aquí **no usamos Pipeline**, y los nombres de la rejilla van **sin** el prefijo `modelo__`.

### 8.1 Árbol de decisión

| Hiperparámetro | Qué controla | Valores que probaremos |
|---|---|---|
| **`max_depth`** | Cuántas preguntas seguidas puede hacer (los "pisos" del árbol). `None` = sin límite | 2 · 3 · 4 · 5 · 6 · `None` |
| **`min_samples_leaf`** | Mínimo de pacientes en cada hoja | 1 · 5 · 10 |
| **`min_samples_split`** 🆕 | Mínimo de pacientes que debe tener un nodo para **poder dividirse**. Evita preguntas sobre grupos muy pequeños | 2 · 10 · 20 |
| **`criterion`** | Fórmula que mide qué tan "revuelto" queda un grupo | `"gini"` · `"entropy"` |
| **`class_weight`** | `"balanced"` da más peso a los malignos | `None` · `"balanced"` |

In [ ]:
# Importando el modelo y la función para dibujarlo
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Creando y ajustando el árbol con valores por defecto (sin Pipeline: no hace falta escalar)
arbol_base = ____(random_state=42)    # ✏️
arbol_base.fit(X_train, y_train)

print('Pisos (profundidad):        ', arbol_base.get_depth())
print('Hojas (decisiones finales): ', arbol_base.get_n_leaves())

In [ ]:
# Calculando las predicciones sobre el conjunto de prueba
y_pred = arbol_base.predict(X_test)

# Imprimiendo el reporte de clasificación
print(classification_report(y_test, y_pred, target_names=clases))

# Construyendo la matriz de confusión
ConfusionMatrixDisplay.____(y_test, y_pred, display_labels=clases, cmap='Blues')    # ✏️
plt.title('Árbol de decisión base')
plt.show()

In [ ]:
# Hiperparámetros a optimizar (sin el prefijo modelo__ porque no hay Pipeline)
rejilla_arbol = {
    "max_depth":         [2, 3, 4, 5, 6, None],        # None = sin límite
    "min_samples_leaf":  [1, 5, 10],
    ____: [2, 10, 20],                  # NUEVO  ✏️
    "criterion":         ["gini", ____],    # ✏️
    "class_weight":      [None, "balanced"],
}
# 6 × 3 × 3 × 2 × 2 = 216 combinaciones (muchas, pero cada árbol tarda milisegundos)

In [ ]:
mejores_modelos = {}

for metrica in metricas:
    busqueda = GridSearchCV(____, ____, scoring=metrica, cv=5, n_jobs=-1)    # ✏️
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.best_estimator_
    print(f'{metrica} >> {busqueda.best_score_:.3f}  {busqueda.best_params_}')

In [ ]:
# Mostrando las matrices de confusión de las 3 versiones optimizadas
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))

for eje, metrica in zip(ejes, metricas):
    y_pred = mejores_modelos[metrica].predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases,
                                            cmap='Blues', colorbar=False, ax=eje)
    eje.set_title(f'Objetivo: {metrica}')

plt.suptitle('Árbol de decisión')
plt.tight_layout()
plt.show()

In [ ]:
# Guardando como finalista la versión optimizada con recall
finalistas['Árbol de decisión'] = mejores_modelos['recall']

In [ ]:
# Dibujando solo los primeros 3 pisos del árbol finalista (completo no cabría en pantalla)
plt.figure(figsize=(22, 9))
plot_tree(finalistas['Árbol de decisión'], feature_names=X.columns, class_names=clases,
          filled=True, rounded=True, impurity=False, max_depth=____, fontsize=9)    # ✏️
plt.show()

> Si el árbol finalista usa `class_weight="balanced"`, los números de `value` son **pacientes ponderados** (cada maligno "vale" más que un benigno), por eso no son enteros. Las cajas **azules** votan *Maligno* y las **naranjas** *Benigno*; entre más intenso el color, más seguro está el árbol.

### 8.2 Random Forest

| Hiperparámetro | Qué controla | Valores que probaremos |
|---|---|---|
| **`n_estimators`** | Cuántos árboles votan | 100 · 300 |
| **`max_features`** | Entre cuántas variables al azar elige cada árbol en cada pregunta. Con 30 variables: `"sqrt"` → 5 · `"log2"` 🆕 → 4 | `"sqrt"` · `"log2"` |
| **`max_depth`** | Igual que en el árbol | 5 · `None` |
| **`min_samples_leaf`** 🆕 | Mínimo de pacientes en cada hoja de cada árbol | 1 · 3 |
| **`class_weight`** | `"balanced"` da más peso a los malignos | `None` · `"balanced"` |

⏱️ Esta búsqueda tarda unos **minutos**: cada combinación entrena cientos de árboles. Ten paciencia.

In [ ]:
# Importando el modelo
from sklearn.ensemble import RandomForestClassifier

# Creando y ajustando el bosque con valores por defecto
bosque_base = ____(random_state=42)    # ✏️
bosque_base.fit(X_train, y_train)

In [ ]:
# Calculando las predicciones sobre el conjunto de prueba
y_pred = bosque_base.predict(X_test)

# Imprimiendo el reporte de clasificación
print(classification_report(y_test, y_pred, target_names=____))    # ✏️

# Construyendo la matriz de confusión
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases, cmap='Blues')
plt.title('Random Forest base')
plt.show()

In [ ]:
# Hiperparámetros a optimizar
rejilla_bosque = {
    "n_estimators":     [100, 300],
    "max_features":     ["sqrt", ____],            # NUEVO: 5 o 4 variables al azar  ✏️
    "max_depth":        [5, None],
    ____: [1, 3],                        # NUEVO  ✏️
    "class_weight":     [None, "balanced"],
}
# 2 × 2 × 2 × 2 × 2 = 32 combinaciones

In [ ]:
mejores_modelos = {}

for metrica in metricas:
    busqueda = GridSearchCV(____, ____, scoring=metrica, cv=5, n_jobs=-1)    # ✏️
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.best_estimator_
    print(f'{metrica} >> {busqueda.best_score_:.3f}  {busqueda.best_params_}')

In [ ]:
# Mostrando las matrices de confusión de las 3 versiones optimizadas
fig, ejes = plt.subplots(1, 3, figsize=(12, 4))

for eje, metrica in zip(ejes, metricas):
    y_pred = mejores_modelos[metrica].predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases,
                                            cmap='Blues', colorbar=False, ax=eje)
    eje.set_title(f'Objetivo: {metrica}')

plt.suptitle('Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Guardando como finalista la versión optimizada con recall
finalistas['Random Forest'] = mejores_modelos['recall']

In [ ]:
# ¿Qué mediciones usó más el bosque para decidir?
importancias = pd.Series(finalistas['Random Forest'].____, index=X.columns)    # ✏️

importancias.sort_values().tail(10).plot.barh(figsize=(8, 5), color='steelblue')
plt.title('Top 10 variables más importantes (Random Forest)')
plt.xlabel('Importancia')
plt.show()

---
## 9 · Comparación final de los cinco finalistas  *(5 puntos)*

Ahora comparamos los 5 finalistas **con los mismos pacientes de prueba**. Completa las fórmulas de sensibilidad y especificidad usando VP, VN, FP y FN.

La tabla queda ordenada de **menos FN a más FN** (y, si hay empate, de menos FP a más FP).

In [ ]:
filas = []

for nombre, modelo in finalistas.items():
    y_pred = modelo.predict(X_test)
    vn, fp, fn, vp = confusion_matrix(y_test, y_pred).ravel()      # los 4 números de la matriz
    filas.append({
        'Modelo':        nombre,
        'Exactitud':     (vp + vn) / len(y_test),
        'Sensibilidad':  ____,             # de los malignos, ¿cuántos detecta?  ✏️
        'Especificidad': ____,             # de los benignos, ¿cuántos descarta bien?  ✏️
        'FN':            fn,
        'FP':            fp,
    })

tabla = pd.DataFrame(filas).sort_values(by=['FN', 'FP'])
tabla.round(3)

---
## 10 · Ajuste del umbral de decisión  *(5 puntos)*

Por defecto, los modelos dicen **"Maligno"** cuando la probabilidad de maligno es **≥ 0.5**. Si **bajamos el umbral**, el modelo se vuelve más "desconfiado": detecta más malignos (**menos FN**), a cambio de más falsas alarmas (**más FP**).

Lo probaremos con la **regresión logística** y el **Random Forest**, que calculan probabilidades con `predict_proba`. (La SVM no las calcula por defecto.)

In [ ]:
umbrales = [0.5, 0.4, 0.3, 0.2, 0.1]

for nombre in ['Regresión logística', 'Random Forest']:
    proba = finalistas[nombre].____(X_test)[:, 1]      # columna 1 = probabilidad de "Maligno"  ✏️
    print(nombre)

    for umbral in umbrales:
        y_pred_u = (proba >= ____).astype(int)                # 1 si supera el umbral, 0 si no  ✏️
        vn, fp, fn, vp = confusion_matrix(y_test, y_pred_u).ravel()
        print(f'   umbral {umbral}:   FN = {fn}   FP = {fp}')
    print()

---
## 11 · Reflexión: ¿cuál es el mejor modelo?  *(30 puntos)*

Responde con tus propias palabras y **apóyate en los números** que obtuviste (matrices, tabla de la sección 9 y umbrales de la sección 10). No basta con decir "es mejor": explica **por qué**.

**Pregunta 1 · Falsos negativos vs. falsos positivos** *(7 pts)*
En este problema, ¿qué significa clínicamente un **falso negativo**? Describe qué le pasaría a esa paciente y compáralo con lo que le pasaría a una paciente con un **falso positivo**. ¿Cuál de los dos errores es más costoso y por qué?

✍️ **Tu respuesta:**

_Escribe aquí tu respuesta…_

**Pregunta 2 · ¿Con qué métrica optimizar?** *(6 pts)*
Elige uno de los modelos y compara sus matrices de confusión con objetivo `precision` y con objetivo `recall`. ¿Qué pasó con los FN y con los FP? ¿Por qué optimizar la precisión **no** es buena idea en este contexto?

✍️ **Tu respuesta:**

_Escribe aquí tu respuesta…_

**Pregunta 3 · El mejor modelo** *(10 pts)*
Con la tabla de la sección 9, ¿cuál fue tu **mejor modelo**? Justifica tu elección con **al menos tres números** (FN, FP, sensibilidad, especificidad…). Si dos modelos empataron en FN, explica cómo desempataste.

✍️ **Tu respuesta:**

_Escribe aquí tu respuesta…_

**Pregunta 4 · ¿Por qué no basta la exactitud?** *(3 pts)*
¿Por qué la exactitud por sí sola no es suficiente para elegir el modelo en este problema? *Pista:* piensa en la proporción de benignos y malignos, y en qué te dice (o no) la exactitud sobre el **tipo** de error.

✍️ **Tu respuesta:**

_Escribe aquí tu respuesta…_

**Pregunta 5 · Umbral y uso real** *(4 pts)*
Según la sección 10, ¿qué umbral recomendarías y cuántos FN y FP tendrías con él? ¿Usarías este modelo para **reemplazar** al patólogo o solo como **apoyo**? Menciona una limitación de este estudio.

✍️ **Tu respuesta:**

_Escribe aquí tu respuesta…_